# Accuracy evaluation and reproducible user sampling

The supplied notebook had a **one-hour time window**, not a random `.sample()` operation.
`sample_users` now samples complete users without replacement; it never samples trajectory points.
The same selection, KNN and pair costs are reused for all k values.

**Reproduction limits:** Table II uses Senegal (`sen`) datasets and k=2, not YJMob.
Figure 5 includes YJMob (`mob`), k=2,5,10, and Table I reports 75 days for mob.
For full-duration experiments use `T_max=None`; a one-hour run is only a smoke test.
This refactor's quadratic DP, corrected RG and optional reciprocal candidates differ
from the original pipeline, so matching metric definitions does not reproduce the published numbers.

The authors' evaluation notebook computes spatial error as the square root of the mean
of the smallest floor(0.95*M) generalized areas, and time error as the mean of the
smallest floor(0.95*M) durations. The two tails are removed independently.
The exact ECDF below uses all generalized samples, including zero-area samples.

In [ ]:
%pip install -e .

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from kscale import (read_and_parse_data, build_KNN, compute_C_tilde,
                    nu_k_pick, generalize_Sigma, calculate_metrics,
                    plot_spatial_accuracy_cdf)

input_path = Path("/mnt/data/k-scale/Data/YJMob100K/yjmob100k-dataset1.csv")
output_path = Path("run_metrics_01")
K = 10
k_values = [2, 5, 10]
workers = 8
backend = "exact"
reciprocal_candidates = True
T_max = 3600  # Preserve your one-hour smoke test; use None for the full data.
sample_users = None  # Optional whole-user sample, e.g. 1000. None keeps all users.
random_state = 42
spatial_thresholds_km2 = [0, 0.1, 0.2, 0.3, 0.5, 1, 2, 4, 7, 10, 25, 50, 100, 500]

## 1. Read and persist Sigma
Each trajectory Parquet contains only x, y, timestamp and site_id.

In [ ]:
Sigma = read_and_parse_data(
    input_path, output_path / "Sigma", source_format="yjmob",
    time_window_seconds=T_max, sample_users=sample_users,
    random_state=random_state,
)
manifest = pd.read_parquet(Sigma / "manifest.parquet")
assert len(manifest) > K, "K must be smaller than the selected user count."
assert max(k_values) <= len(manifest), "Not enough users for the requested k."
assert K >= max(k_values) - 1, "Increase K to support all requested k values."
manifest[["N_i"]].describe()

## 2. Build KNN from CM, RG and D (Sections III-B/C)

In [ ]:
KNN = build_KNN(
    Sigma, output_path / "KNN", K=K, backend=backend,
    workers=workers if backend == "exact" else 1, batch_size=100,
)

## 3. Compute C_tilde, select A, and generalize Sigma
These substeps are independent and can be resumed using their saved paths.

In [ ]:
C_tilde = compute_C_tilde(
    Sigma, KNN, output_path / "C_tilde", workers=workers,
    reciprocal_candidates=reciprocal_candidates,
)

In [ ]:
generalized_paths = {}
for k in k_values:
    run_path = output_path / f"k_{k}"
    A = nu_k_pick(C_tilde, run_path / "A", k=k)
    generalized_paths[k] = generalize_Sigma(
        Sigma, A, run_path / "Sigma_tilde", workers=workers,
    )

## 4. Calculate accuracy metrics (Section IV)
This step reads saved outputs; it does not rerun K-Merge.

In [ ]:
metric_paths = {}
summaries = []
for k, Sigma_tilde in generalized_paths.items():
    metric_path = output_path / f"k_{k}" / "metrics"
    summary = calculate_metrics(
        Sigma, Sigma_tilde, metric_path, verify_containment=True,
        spatial_thresholds_km2=spatial_thresholds_km2, trim_fraction=0.95,
    )
    metric_paths[f"k={k}"] = metric_path
    summaries.append(pd.read_csv(metric_path / "paper_metrics.csv"))
comparison = pd.concat(summaries, ignore_index=True)
comparison.to_csv(output_path / "paper_metrics_comparison.csv", index=False)
comparison

## Spatial accuracy CDF
The main curve uses F(a)=count(area <= a)/M. No trimming or histogram normalization is applied.

In [ ]:
ax = plot_spatial_accuracy_cdf(metric_paths, convention="ecdf")
ax.figure.savefig(output_path / "spatial_accuracy_cdf.png", dpi=200, bbox_inches="tight")
ax.figure.savefig(output_path / "spatial_accuracy_cdf.pdf", bbox_inches="tight")
plt.show()

In [ ]:
threshold_tables = []
for label, metric_path in metric_paths.items():
    table = pd.read_csv(metric_path / "spatial_accuracy_thresholds.csv")
    table["run"] = label
    threshold_tables.append(table)
pd.concat(threshold_tables).pivot(index="spatial_accuracy_km2", columns="run", values="cdf")

## Optional: original histogram convention
This compatibility curve excludes areas below 0.1 km² from its denominator and uses strict internal upper edges. It is not the full empirical CDF. Zero eligible samples produce an undefined curve.

In [ ]:
ax = plot_spatial_accuracy_cdf(metric_paths, convention="repository_histogram")
ax.figure.savefig(output_path / "spatial_accuracy_repository_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

## Recalculate metrics without rerunning anonymization
If you already have Sigma and Sigma_tilde, call calculate_metrics with their saved paths and a **new metrics output directory**. Only a change to sampling or the time window requires rebuilding the earlier stages.

## 5. Privacy attacks A(c, beta)
Run after generalization. See README_ATTACKS.md. No re-anonymization is required.

In [ ]:
from kscale import simulate_attacks, plot_attack_heatmap
attack_paths = {}
for k, release in generalized_paths.items():
    attack_paths[k] = simulate_attacks(
        Sigma, release, output_path / f"k_{k}" / "attacks_01",
        c_values=[1,2,3,5], beta_values=[1,2,"all"],
        trials=100, seed=42,
    )
